## 1. Business Context

SportsStats a besoin d'une couche analytique pour centraliser et nettoyer ses données éparpillées afin de piloter sa stratégie ( tout le monde se base sur les mêmes chiffres dans l'entreprise ) et de prendre des décisions rapides sans ralentir ses outils de production quotidiens. Ainsi la couche analytique sépare les données opérationnelles brutes de l'analyse métier, en s'assurant que tous les utilisateurs travaillent avec des données cohérentes, validées et réutilisables

## 2. Notebook Objective
 
 Dans ce notebook, je vais créer une vue analytique réutilisable. Ce qui va me permettre d'avoir un modèle de données pré-calculé, pré-organisé et standardisé qui pourra être partagé et réutilisé par plusieurs rapports, utilisateurs ou outils de business intelligence. Cette vue analytique réutilisable sera créée en combinant des données olympiques nettoyées avec des fonctionnalités commerciales conçues. Elle deviendra la source unique de vérité pour les analyses SQL, les tableaux de bord Power BI et les travaux analytiques futurs. 

## 3. Why an Analytical View?
Considérons les deux schémas suivant:    Raw Data  ──>  Business Analytics et  Raw Data  ──>  Cleaning  ──>  Analytical View  ──>  Business Analytics.

Dans le premier scénario, on branche directement l'outil de production à nos données brutes de production. Le problème est que les données brutes sont sales et fragmentées. Ainsi pour obtenir un KPI par exemple, l'analyste doit écrire manuellemnt des formules complexes. Et en conséquence, sa logique métier (intelligence) est stockée dans le rapport lui même. S'il se trompe dans se formule Excel ou son code PowerBi le rapport est faussé. Et pire encore : si un autre collègue crée un autre rapport sur la même donnée brute, il appliquera une logique légèrement différente, et vous vous retrouverez avec deux rapports officiels qui se contredisent.

Dans le second scénario, on introduit une étape importante dans le processus: on extrait la complexité technique du rapport et on le centalise. L'étape cleaning permet le traitement et la correction des incohérence dans les données. L'étape anlytical view permt de créer une vue analytics qui contient toute la logique métier validée par l'entreprise. Et les outils de production viennent simplement se brancher à cette vue. Ce qui permet d'éviter des problèmes comme la duplication de logique métier qui se traduit par l'exemple suivant: trois analystes calculent le BMI chacun dans Power BI et on obtient trois calculs différents. Par contre Si le BMI existe déjà dans la vue analytique, tout le monde utilise exactement le même calcul.


## 4. Analytical Architecture

```text
       [ Sources de Données ] 
     
                 │
                 ▼
          ┌─────────────┐
          │ Raw Dataset │ (Données brutes)
          └──────┬──────┘
                 │
                 ▼
 ┌───────────────────────────────┐
 │    Data Quality Assessment    │ (Contrôle automatique : formats, valeurs nulles)
 └───────────────┬───────────────┘
                 │
                 ▼
          ┌─────────────┐
          │Data Cleaning│ (Déduplication, standardisation, filtres)
          └──────┬──────┘
                 │
                 ▼
   ┌───────────────────────────┐
   │    Feature Engineering    │ ( business features )
   └─────────────┬─────────────┘
                 │
                 ▼
        ┌─────────────────┐
        │ Analytical View │ (La vue unique)
        └────────┬────────┘
                 │
       ┌─────────┴─────────┐
       ▼                   ▼
  ┌──────────┐       ┌──────────────┐
  │ Power BI │       │ad-hoc Analysis │ (Exploration, Data Science)
  └──────────┘       └──────────────┘

## 5. Analytical View Design

|Column           | Source        |Type      |Business Purpose                      |  
|-----------------|---------------|----------|-------------------------------------| 
| ID              |athlete_events |Integer   | Athlete identifier            
| Name            |athlete_events | String   | Athlete name                         
| Sport           | athlete_events | string | Sport category 
 | Region         | noc_regions | string | Country analysis  
  | BMI  | Engineered | float | Physical performance analysis 
   | Event | athlete_events | String |  Games
 | Medal_Status | Engineered | String | Medal analysis
  | Participation_Count | Engineered | Integer | Experience indicator
   | Experience_Level | Engineered | string | Athelete Seniority
| BMI_Category | Engineered | String | Athlete BMI segmentation
 | Era | Engineered | String | Games period segmentation
| Medal | athlete_events | String | Medal type
| First_Olympic_Year | Engineered | Integer | athlete first olypic year
 | Sex | Athlete_events | String | Athlete sex
 | Year | athlete_events | integer | games year
  | Season | athlete_events | string | games sason
   | Team | athlete_events | string | athlete team 
| Is_Medalist | Engineered | int | athlete status


In [0]:
DESCRIBE vw_olympics_analytics

col_name,data_type,comment
ID,bigint,null
Name,string,null
Sex,string,null
Age,int,null
Height,int,null
Weight,int,null
Team,string,null
NOC,string,null
Games,string,null
Year,bigint,null


## 6. Build the Analytical View

In [0]:
CREATE OR REPLACE VIEW vw_olympics_analytics AS

--=============================================
-- Step 1: Load raw Olympic athlete dataset
--============================================

WITH base_data AS (
    SELECT *
    FROM athlete_events
),

--============================================
-- Step 2: Join country and region information
--============================================
joined_regions AS (
    SELECT b.*,
      n.region AS Region
    FROM base_data b
    LEFT JOIN noc_regions n
    ON b.NOC = n.NOC
),

--================================================
-- Step 3: Clean the data
--================================================
cleaned_data AS (
  SELECT ID,
     Name,
     Sex,
     try_cast(Age AS INT) AS Age,
     try_cast(Height AS INT) AS Height,
     try_cast(Weight AS INT) AS Weight,
     Team,
     NOC,
     Games,
     Year,
     Season,
     initcap(City) AS City,
     Sport,
     Event,
     Medal,
     Region
  FROM joined_regions
),

--=====================================================================
-- Step 4: Build the variables that give value to the analytical model
--====================================================================
engineered_features AS (
  SELECT ID,
       Name,
       Sex,
       Age,
       Height,
       Weight,
       Team,
       NOC,
       Games,
       Year,
       Season,
       City,
       Sport,
       Event,
       Medal,
       Region,
       -- Medal features
       CASE WHEN Medal IS NULL OR Medal = '' OR Medal = 'NA' THEN 'Non-Medalist' 
             ELSE 'Medalist' END AS Medal_Status,
       CASE WHEN Medal IS NULL OR Medal = '' OR Medal = 'NA' THEN 0 
             ELSE 1 END AS Is_Medalist,
        -- Physical features
        round(Weight / POWER(NULLIF(Height,0) / 100.0, 2),2) AS BMI,
        CASE WHEN round(Weight / POWER(NULLIF(Height,0) / 100.0, 2),2)< 18.5 THEN 'Underweight'
            WHEN round(Weight / POWER(NULLIF(Height,0) / 100.0, 2),2) BETWEEN 18.5 AND 24.9 THEN 'Normal'
            WHEN round(Weight / POWER(NULLIF(Height,0) / 100.0, 2),2) BETWEEN 25 AND 29.9 THEN 'Overweight'
            WHEN round(Weight / POWER(NULLIF(Height,0) / 100.0, 2),2)>= 30 THEN 'Obesity'
            WHEN Weight IS NULL AND Height IS NULL THEN 'Missing Weight and Height'
            WHEN Weight = 0 THEN 'Weight = 0'
            WHEN Weight IS NULL THEN ' Missing Weight '
            WHEN Height IS NULL THEN 'Missing Height ' 
            END AS BMI_Category,
        -- Participation Features
        COUNT (*)OVER(PARTITION BY ID) AS Participation_Count,
        Min(Year)OVER(PARTITION BY ID) AS First_Olympic_Year,
        CASE WHEN COUNT (*)OVER(PARTITION BY ID) = 1 THEN 'Rookie'
             WHEN COUNT (*)OVER(PARTITION BY ID) BETWEEN 2 AND 3 THEN 'Experienced'
             WHEN COUNT (*)OVER(PARTITION BY ID)>= 4 THEN 'Veteran'
             END AS Experience_Level  
    FROM cleaned_data
    ),
  
--=======================================================================
--Step 5: Create the final dataset
--=======================================================================
final_dataset AS(
SELECT
-- Athlete Information
ID,
Name,
Sex,
Age,
Height,
Weight,
-- Olympic Information
Team,
NOC,
Games,
Year,
Season,
City,
Region,
Sport, 
Event,
-- Medal Information
 Medal,
Medal_Status,
Is_Medalist,
 -- Physical Features
BMI,
BMI_Category,
-- Participation Features
Participation_Count,
First_Olympic_Year,
Experience_Level

FROM engineered_features
)
SELECT *
FROM final_dataset;

 





## 6. Validate the Analytical View

In [0]:
--Le résultat est 271116 et est identique au nombre de lignes attendu après le nettoyage.
SELECT COUNT(*) AS Total_Rows
FROM vw_olympics_analytics;


Total_Rows
271116


In [0]:
--Les colonnes essentielles ne présentent pas d'anomalies.
SELECT

COUNT(*) AS Total,

COUNT(ID) AS ID_Not_Null,

COUNT(Name) AS Name_Not_Null,

COUNT(Year) AS Year_Not_Null,

COUNT(Sport) AS Sport_Not_Null

FROM vw_olympics_analytics;


Total,ID_Not_Null,Name_Not_Null,Year_Not_Null,Sport_Not_Null
271116,271116,271116,271116,271116


In [0]:
--Il existe bien des Medalist et Non-Medalist
SELECT

Medal_Status,

COUNT(*) AS Total

FROM vw_olympics_analytics

GROUP BY Medal_Status;

Medal_Status,Total
Non-Medalist,231333
Medalist,39783


In [0]:
-- Aucune catégorie n'est anormalement vide.
SELECT

BMI_Category,

COUNT(*) AS Total

FROM vw_olympics_analytics
GROUP BY BMI_Category
ORDER BY Total DESC;

BMI_Category,Total
Normal,161702
Missing Weight and Height,58875
Overweight,29632
Underweight,8023
Missing Weight,5058
Obesity,4327
null,2203
Missing Height,1296


In [0]:
--La logique métier produit des résultats réalistes.
SELECT

Experience_Level,

COUNT(*) AS Total

FROM vw_olympics_analytics

GROUP BY Experience_Level;

Experience_Level,Total
Rookie,77901
Veteran,98018
Experienced,95197


In [0]:
-- Toutes les valeurs sont cohérentes
SELECT

MIN(Age),

MAX(Age),

MIN(BMI),

MAX(BMI)

FROM vw_olympics_analytics;

MIN(Age),MAX(Age),MIN(BMI),MAX(BMI)
10,97,8.36,63.9


In [0]:
-- Tout est correcte
SELECT *
FROM vw_olympics_analytics
LIMIT 20;

ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Region,Sport,Event,Medal,Medal_Status,Is_Medalist,BMI,BMI_Category,Participation_Count,First_Olympic_Year,Experience_Level
9,Antti Sami Aalto,M,26,186,96,Finland,FIN,2002 Winter,2002,Winter,Salt Lake City,Finland,Ice Hockey,Ice Hockey Men's Ice Hockey,NA,Non-Medalist,0,27.75,Overweight,1,2002,Rookie
11,Jorma Ilmari Aalto,M,22,182,null,Finland,FIN,1980 Winter,1980,Winter,Lake Placid,Finland,Cross Country Skiing,Cross Country Skiing Men's 30 kilometres,NA,Non-Medalist,0,null,Missing Weight,1,1980,Rookie
28,Jan-Erik Aarberg,M,43,170,77,Norway,NOR,1968 Summer,1968,Summer,Mexico City,Norway,Sailing,Sailing Mixed Three Person Keelboat,NA,Non-Medalist,0,26.64,Overweight,2,1968,Experienced
28,Jan-Erik Aarberg,M,47,170,77,Norway,NOR,1972 Summer,1972,Summer,Munich,Norway,Sailing,Sailing Mixed Three Person Keelboat,NA,Non-Medalist,0,26.64,Overweight,2,1968,Experienced
29,Willemien Aardenburg,F,22,null,null,Netherlands,NED,1988 Summer,1988,Summer,Seoul,Netherlands,Hockey,Hockey Women's Hockey,Bronze,Medalist,1,null,Missing Weight and Height,1,1988,Rookie
30,Pepijn Aardewijn,M,26,189,72,Netherlands,NED,1996 Summer,1996,Summer,Atlanta,Netherlands,Rowing,Rowing Men's Lightweight Double Sculls,Silver,Medalist,1,20.16,Normal,2,1996,Experienced
30,Pepijn Aardewijn,M,30,189,72,Netherlands,NED,2000 Summer,2000,Summer,Sydney,Netherlands,Rowing,Rowing Men's Lightweight Double Sculls,NA,Non-Medalist,0,20.16,Normal,2,1996,Experienced
32,Olav Augunson Aarnes,M,23,null,null,Norway,NOR,1912 Summer,1912,Summer,Stockholm,Norway,Athletics,Athletics Men's High Jump,NA,Non-Medalist,0,null,Missing Weight and Height,1,1912,Rookie
33,Mika Lauri Aarnikka,M,24,187,76,Finland,FIN,1992 Summer,1992,Summer,Barcelona,Finland,Sailing,Sailing Men's Two Person Dinghy,NA,Non-Medalist,0,21.73,Normal,2,1992,Experienced
33,Mika Lauri Aarnikka,M,28,187,76,Finland,FIN,1996 Summer,1996,Summer,Atlanta,Finland,Sailing,Sailing Men's Two Person Dinghy,NA,Non-Medalist,0,21.73,Normal,2,1992,Experienced
